In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries loaded successfully!")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

✓ All libraries loaded successfully!
Numpy version: 2.0.2
Pandas version: 2.2.2


In [5]:
# Generate synthetic customer churn dataset
X, y = make_classification(
    n_samples=10000,          # 10,000 customers
    n_features=20,            # 20 behavioral features
    n_informative=15,         # 15 features actually predict churn
    n_redundant=5,            # 5 features are redundant
    n_classes=2,              # Binary: churn (1) or retain (0)
    weights=[0.95, 0.05],     # 5% churn rate
    flip_y=0.01,              # 1% label noise (realistic)
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,            # 30% test set
    stratify=y,               # Preserve 5% churn rate in both sets
    random_state=42
)

print("✓ Dataset generated successfully!")
print("\n" + "#"*70)

print(f"Training set: {X_train.shape[0]:,} customers")
print(f"Test set: {X_test.shape[0]:,} customers")

print("\nChurn Distribution (Test Set):")
print(f"  Retained (0): {np.sum(y_test == 0):,} customers "
      f"({np.mean(y_test == 0)*100:.1f}%)")
print(f"  Churned  (1): {np.sum(y_test == 1):,} customers "
      f"({np.mean(y_test == 1)*100:.1f}%)")

print("\nFeatures: 20 (usage patterns, billing, support interactions)")
print("#"*70)

✓ Dataset generated successfully!

######################################################################
Training set: 7,000 customers
Test set: 3,000 customers

Churn Distribution (Test Set):
  Retained (0): 2,838 customers (94.6%)
  Churned  (1): 162 customers (5.4%)

Features: 20 (usage patterns, billing, support interactions)
######################################################################


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(
    max_iter = 1000,          # Sufficient iterations for convergence
    random_state = 42,        # Reproducibility
    class_weight = 'balanced' # Handle 5% imbalance
)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [8]:
y_proba = model.predict_proba(X_test_scaled)[:,1]
y_pred = model.predict(X_test_scaled)

accuracy = model.score(X_test_scaled, y_test)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Probability scores generated")
print(f"Min probability: {y_proba.min():.4f}")
print(f"Max probability: {y_proba.max():.4f}")
print(f"Mean probability: {y_proba.mean():.4f}")

Accuracy: 0.8347
Recall: 0.7469
Precision: 0.2101
F1 Score: 0.3279
Probability scores generated
Min probability: 0.0032
Max probability: 0.9965
Mean probability: 0.3058


In [9]:
# Show example predictions
print("\nExample Customer Scores:")
print("Customer ID | Churn Prob | Actual | Interpretation")
print("-"*70)

for i in (0, 100, 500, 1000, 2000):
    prob = y_proba[i]
    actual = "Churned" if y_test[i] == 1 else "Retained"
    risk = "HIGH" if prob > 0.7 else ("MEDIUM" if prob > 0.3 else "LOW")

    print(f"{i:11d} | {prob:10.4f} | {actual:7s} | {risk} risk")


Example Customer Scores:
Customer ID | Churn Prob | Actual | Interpretation
----------------------------------------------------------------------
          0 |     0.0950 | Retained | LOW risk
        100 |     0.5386 | Retained | MEDIUM risk
        500 |     0.3253 | Retained | MEDIUM risk
       1000 |     0.2892 | Retained | LOW risk
       2000 |     0.2264 | Retained | LOW risk


In [10]:
print("LIFT calculation: Top 10% of customers")
sorted_indices = np.argsort(y_proba)[::-1]
y_test_sorted = y_test[sorted_indices]

top_ten_percent_size = int(len(y_test) * 0.10)
top_ten_percent = y_test_sorted[:top_ten_percent_size]

overall_churn_rate = y_test.mean()
top_ten_churn_rate = top_ten_percent.mean()
print(overall_churn_rate, top_ten_churn_rate)
lift_top_ten = top_ten_churn_rate / overall_churn_rate
lift_top_ten

LIFT calculation: Top 10% of customers
0.054 0.36333333333333334


np.float64(6.728395061728396)

In [12]:
print("\nPopulation:", len(y_test), "customers")
print(f"Top 10% size: {top_ten_percent_size:,} customers")

print("\nRANDOM APPROACH:")
print(f"  Expected churners in random 10%: {int(len(y_test) * 0.10 * overall_churn_rate):,}")
print(f"  Response rate: {overall_churn_rate*100:.2f}%")

print("\nMODEL APPROACH:")
print(f"  Actual churners in top 10%: {int(top_ten_percent.sum()):,}")
print(f"  Response rate: {top_ten_churn_rate*100:.2f}%")

print("\nLIFT:")
print(f"  Lift (top_10): {lift_top_ten:.2f}x better than random")

print("\nBUSINESS TRANSLATION:")
print(f"  ➤ By using our model to identify the top 10% highest‑risk customers,")
print(f"    we catch {lift_top_ten:.1f}× more churners than calling random customers.")

print(f"\n✅ Same budget, {lift_top_ten:.1f}× better results.")
print("#"*70)


Population: 3000 customers
Top 10% size: 300 customers

RANDOM APPROACH:
  Expected churners in random 10%: 16
  Response rate: 5.40%

MODEL APPROACH:
  Actual churners in top 10%: 109
  Response rate: 36.33%

LIFT:
  Lift (top_10): 6.73x better than random

BUSINESS TRANSLATION:
  ➤ By using our model to identify the top 10% highest‑risk customers,
    we catch 6.7× more churners than calling random customers.

✅ Same budget, 6.7× better results.
######################################################################


In [14]:
def calculate_cumulative_lift(y_true, y_proba):
    # Sort predictions by probability in descending order
    sorted_indices = np.argsort(y_proba)[::-1]
    y_sorted = y_true[sorted_indices]

    # Calculate overall response rate (random baseline)
    overall_rate = y_true.mean()

    percentiles = []
    lifts = []

    # Calculate lift at each decile
    for pct in range(10, 101, 10):
        cutoff = int(len(y_sorted) * pct / 100)
        segment = y_sorted[:cutoff]
        segment_rate = segment.mean()

        lift = segment_rate / overall_rate

        percentiles.append(pct)
        lifts.append(lift)

    return percentiles, lifts

# Calculate lift curve
percentiles, lifts = calculate_cumulative_lift(y_test, y_proba)

print("CUMULATIVE LIFT BY DECILE")
print("-" * 70)
print("Percentile | Lift | Interpretation")
print("-" * 70)

for pct, lift in zip(percentiles, lifts):
    interpretation = "Excellent" if lift > 3 else ("Good" if lift > 2 else "Fair")
    print(f"{pct:>10d}% | {lift:5.2f} | {interpretation}")

CUMULATIVE LIFT BY DECILE
----------------------------------------------------------------------
Percentile | Lift | Interpretation
----------------------------------------------------------------------
        10% |  6.73 | Excellent
        20% |  3.80 | Excellent
        30% |  2.63 | Good
        40% |  2.08 | Good
        50% |  1.75 | Fair
        60% |  1.51 | Fair
        70% |  1.34 | Fair
        80% |  1.20 | Fair
        90% |  1.11 | Fair
       100% |  1.00 | Fair
